# bias-correction-divide — ex1: bias-correct an Adam moment: m_hat = m / (1 - beta**t)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `bias-correction-divide`. Running the final beacon cell reports progress against the `Optimizer: Adam bias-correction divide` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: Adam bias-correction divide` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`bias-correction-divide`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "bias-correction-divide"
DD_SUBTOPIC = "Optimizer: Adam bias-correction divide"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Adam bias correction `m_hat = m / (1 - beta1**t)` — quick refresher

Both Adam moments start at zero. The EMA recurrence `m_t = beta1 * m_{t-1} + (1 - beta1) * g_t` therefore drags toward zero in the FIRST FEW steps even when the true gradient is far from zero. Concretely, for constant `g`:

```
m_1 = (1 - beta1) * g
m_2 = (1 - beta1**2) * g
...
m_t = (1 - beta1**t) * g
```

So `m_t / g = 1 - beta1**t` — the EMA is BIASED toward zero by a factor of `(1 - beta1**t)`. The bias correction divides it out:

```
m_hat = m / (1 - beta1**t)
v_hat = v / (1 - beta2**t)
```

**Why this matters.** Without the correction, the first few steps of Adam would take suspiciously tiny updates — `m` is small not because the gradient is small, but because the EMA hasn't warmed up. The correction makes step 1 take a full-magnitude update.

**`t` is the STEP COUNTER**, not a tensor. It starts at 1 (not 0) and is incremented after each optimizer step. `beta1**t` shrinks toward 0 with `t`, so the divisor `(1 - beta1**t)` grows toward 1 — the correction fades to a no-op as training progresses.

### Exercise 1 — bias-correct an Adam moment: m_hat = m / (1 - beta**t)

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the Adam bias-correction divide `m_hat = m / (1 - beta**t)` to undo the zero-initialization bias of an EMA buffer at step `t`.
> Keywords: bias-correction, adam, warmup, step-counter
> ```

**KCs targeted:** `bias-correction-divide-by-one-minus-beta-power-t`, `step-counter-1-based-for-bias-correction`

Implement `ex1_bias_correct(m, beta, t_step)`. The two-argument bias-correction divide from Adam.

1. Compute the correction factor `1 - beta ** t_step`. This is a Python scalar (not a tensor).
2. Return `m / correction`. Do NOT mutate `m` in place — return a new tensor.

Inputs:
- `m`: EMA buffer (tensor, any shape).
- `beta`: float decay coefficient (e.g. 0.9 for first moment, 0.999 for second moment).
- `t_step`: int >= 1, the current step (1-based).

Output: bias-corrected `m_hat`, same shape as `m`.

**Critical:** `t_step` is 1-based, not 0-based. At step 1, the correction is `1 / (1 - beta)`, which is BIG — it amplifies the small warmup-EMA back to full magnitude. If you pass `t_step=0` you get a division by zero.

In [ ]:
def ex1_bias_correct(m: Tensor, beta: float, t_step: int) -> Tensor:
    """Return m / (1 - beta**t_step)."""
    raise NotImplementedError()


def _test_ex1():
    # === Step 1 with beta=0.9 and constant g => m = 0.1 * g.
    # Bias correction should recover m_hat == g.
    g = t.tensor([1.0, 2.0, 3.0, 4.0])
    beta1 = 0.9
    m_step1 = (1 - beta1) * g       # what the EMA produces at step 1
    m_hat = ex1_bias_correct(m_step1, beta1, t_step=1)
    assert t.allclose(m_hat, g, atol=1e-6), (
        f'step 1 with constant g should recover g exactly; got {m_hat}, expected {g}; '
        f'check formula: m / (1 - beta**t)'
    )

    # === Step 5 with beta=0.999 (second-moment style) — correction approaches 1 as t grows.
    beta2 = 0.999
    v = t.tensor([0.5, 1.0, 2.0])
    v_hat = ex1_bias_correct(v, beta2, t_step=5)
    expected_v_hat = v / (1 - beta2 ** 5)
    assert t.allclose(v_hat, expected_v_hat, atol=1e-6), (
        f'step 5 with beta=0.999: expected {expected_v_hat}, got {v_hat}'
    )
    # The correction divisor at step 5 with beta=0.999 is tiny → v_hat is HUGE.
    assert (v_hat > v).all(), 'bias-corrected v should be larger than raw v early in training'

    # === As t -> inf, correction -> 1 → m_hat -> m ===
    m_big_t = ex1_bias_correct(t.tensor([1.0, 1.0]), beta=0.9, t_step=10000)
    assert t.allclose(m_big_t, t.tensor([1.0, 1.0]), atol=1e-6), (
        f'large t: m_hat should approach m; got {m_big_t}'
    )

    # === Multi-dim shape preservation ===
    m_2d = t.randn(4, 5)
    out_2d = ex1_bias_correct(m_2d, beta=0.9, t_step=3)
    assert out_2d.shape == (4, 5)
    expected_2d = m_2d / (1 - 0.9 ** 3)
    assert t.allclose(out_2d, expected_2d, atol=1e-6)

    # === Input m must NOT be mutated ===
    m_in = t.tensor([1.0, 2.0, 3.0])
    m_snap = m_in.clone()
    _ = ex1_bias_correct(m_in, beta=0.9, t_step=1)
    assert t.equal(m_in, m_snap), (
        f'input m was mutated (now {m_in}, was {m_snap}); '
        f'this fold must be out-of-place'
    )

    # === Step-1 closed-form sanity: m_hat[i] should equal g[i] exactly when EMA started from 0.
    # Round-trip: run the EMA recurrence then bias-correct → recover g.
    for beta_test in [0.5, 0.9, 0.999]:
        for step_test in [1, 2, 5, 10]:
            # Closed form for constant g across step_test steps: m = (1 - beta**step_test) * g
            m_synth = (1 - beta_test ** step_test) * g
            recovered = ex1_bias_correct(m_synth, beta_test, step_test)
            assert t.allclose(recovered, g, atol=1e-5), (
                f'round-trip failed for beta={beta_test}, t={step_test}: got {recovered}, expected {g}'
            )
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_bias_correct(m, beta, t_step):
    return m / (1 - beta ** t_step)
```

**Why `beta ** t_step` (Python power) not `t.pow(beta, t_step)`.** `beta` is a Python float and `t_step` is a Python int — `**` evaluates on the host as a plain scalar. Wrapping it in a tensor op would create a tiny GPU synchronization for nothing. Adam's reference impl does the scalar.

**Why this is its own atom.** The bias correction is a ONE-LINE divide, but it's where two-thirds of Adam-from-scratch bugs live: passing `t=0`, forgetting to update `t` between steps, applying the correction to the wrong moment (e.g. `m / (1 - beta2**t)`). Isolating it as its own drill lets you nail the placement of `t_step` and the `(1 - beta**t)` denominator before composing it into the full Adam impl.

**Aside: AdamW skips bias correction.** Some literature claims AdamW doesn't bias-correct; the reference HuggingFace AdamW implementation DOES still apply it — only the weight-decay term is decoupled, not the bias correction. Don't be tricked by old blog posts.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()